# 85 — Load, Test & Enhance: 84-Simulate10Next_Conqueror_Supplier_fixed

Test notebook for `StrategyPipeline` from `84-Simulate10Next_Conqueror_Supplier_fixed.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

| Test | Setup |
|------|-------|
| Test 01 | 2 Conquerors, 1 Enemy |
| Test 02 | 2 Conquerors, 2 Neutrals |
| Test 04 | 1 Supplier, 2 Conquerors, 1 Enemy |

In [1]:
%run 84-Simulate10Next_Conqueror_Supplier_fixed.py

In [2]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

# Testing new function

In [3]:
def _04_score_and_decide(attacks_with_angle: pd.DataFrame, player_id: int) -> list:
    if attacks_with_angle.empty:
        return []

    moves = []

    # Comet evasion
    awa_comets = attacks_with_angle[attacks_with_angle["nature_src"] == "comet"]
    if not awa_comets.empty:
        x_off = (awa_comets["x_src"] - GameConfig.CENTER).abs().max() or 0
        y_off = (awa_comets["y_src"] - GameConfig.CENTER).abs().max() or 0
        if max(x_off, y_off) > 45:
            moves += (
                awa_comets[awa_comets["ships_sent"] <= awa_comets["ships_min"]]
                .sort_values(["ships_sent", "step"], ascending=[False, True])
                .groupby("id_src", sort=False)
                .first()
                .reset_index()
                [["id_src", "final_angle", "ships_sent"]]
                .values.tolist()
            )
            id_to_avoid = awa_comets["id_src"].unique().tolist()
            attacks_with_angle = attacks_with_angle[~attacks_with_angle["id_src"].isin(id_to_avoid)]

    # Top-5 targets per source planet (cheapest by step then ships_sent)
    top5_ids = (
        attacks_with_angle
        .sort_values(["step", "ships_sent"])
        .groupby(["id_src", "id"], sort=False)
        .first()
        .reset_index()
        .sort_values(["step", "ships_sent"])
        .groupby("id_src", sort=False)
        .head(5)
        [["id_src", "id"]]
        .assign(is_top5=True)
    )

    # Source planet IDs owned by player (id_src is already filtered to player's planets)
    mine_src_ids = set(attacks_with_angle["id_src"].unique())

    # Classify each source as Supplier (all top5 targets are own planets) or Conqueror
    top5_with_mine = top5_ids.copy()
    top5_with_mine["target_is_mine"] = top5_with_mine["id"].isin(mine_src_ids)
    src_nature = (
        top5_with_mine
        .groupby("id_src")
        .agg(mine_count=("target_is_mine", "sum"), total_count=("target_is_mine", "count"))
        .reset_index()
    )
    src_nature["status"] = np.where(
        src_nature["mine_count"] == src_nature["total_count"], "Supplier", "Conqueror"
    )
    conqueror_ids = set(src_nature.loc[src_nature["status"] == "Conqueror", "id_src"])
    supplier_ids  = set(src_nature.loc[src_nature["status"] == "Supplier",  "id_src"])

    # ── Conqueror: attack enemy/neutral planets ──────────────────────────────
    attacks_conqueror = pd.DataFrame()
    conqueror_needs = None
    if conqueror_ids:

        _c_1_or_2 = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(conqueror_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .query("is_top5")
            .loc[lambda d: d["owner"] != player_id]
            .assign(ships_needed=lambda d: np.where(
                d["owner"] == -1, d["ships"], d["ships"] + d["production"]
            ))
            # .loc[lambda d:
            #     (d["ships_needed"] + 1 <= d["ships_sent"]) &
            #     (d["ships_sent"] <= d["ships_needed"] + d["production_src"] + 1)
            # ]
            # .sort_values(["step", "ships_sent"])
            # .groupby(["id_src", "id"], sort=False).first().reset_index()
            # .assign(time_cost=lambda d: d["ships_needed"] / d["production_src"])
        )


        _c = (
            _c_1_or_2
            # attacks_with_angle[attacks_with_angle["id_src"].isin(conqueror_ids)]
            # .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            # .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            # .query("is_top5")
            # .loc[lambda d: d["owner"] != player_id]
            # .assign(ships_needed=lambda d: np.where(
            #     d["owner"] == -1, d["ships"], d["ships"] + d["production"]
            # ))
            .loc[lambda d:
                (d["ships_needed"] + 1 <= d["ships_sent"]) &
                (d["ships_sent"] <= d["ships_needed"] + d["production_src"] + 1)
            ]
            .sort_values(["step", "ships_sent"])
            .groupby(["id_src", "id"], sort=False).first().reset_index()
            .assign(time_cost=lambda d: d["ships_needed"] / d["production_src"])
        )
        if not _c.empty:
            conqueror_needs = (
                _c
                .groupby("id_src", sort=False)
                .agg(
                    ship_min=("ships_min", "min"),
                    all_need=("ships_sent", "sum"),
                    lowest_need=("ships_sent", "min"),
                    nb_need=("ships_sent", "count"),
                )
            )
            attacks_conqueror = (
                _c
                .assign(
                    total_time_cost=_c.groupby("id_src")["time_cost"].transform("sum")
                ).assign(
                    score=lambda d: (
                        (d["total_time_cost"] - d["time_cost"] - d["step_diff"]) * d["production"]
                    )
                )
                # .loc[lambda d: d["score"] > 0]
                .sort_values("score", ascending=False)
                .groupby("id_src", sort=False).first().reset_index()
                .loc[lambda d: d["ships_sent"] <= d["ships_min"]]
            )
        if not _c_1_or_2.empty:
            attacks_conqueror_2 = (
                _c_1_or_2
                .merge(
                    _c_1_or_2,
                    on="id",
                    how="inner",
                    suffixes=("", "_2"), # _2 is the ship to be sent later
                )
                .query("id_src != id_src_2")
                .query("step < step_2")
                .loc[lambda d:
                    (np.maximum(d["ships_needed"], d["ships_needed_2"]) + 1 <= d["ships_sent"] + d["ships_sent_2"]) &
                    (d["ships_sent"] + d["ships_sent_2"] <= np.maximum(d["ships_needed"], d["ships_needed_2"]) + d["production_src_2"] + 1)
                ]
                .loc[lambda d: d["ships_sent"] <= d["ships_min"]]
                .loc[lambda d: d["ships_sent_2"] <= d["ships_min_2"] +  d["production_src_2"]]
                .sort_values(["step_2", "ships_sent_2"])
                .groupby(["id_src", "id"], sort=False).first().reset_index()
                .sort_values(["step_2", "ships_sent_2"])
                .pipe(lambda d: d.head(1) if d is not None and not d.empty else None)
            )


    # ── Supplier: reinforce own planets ─────────────────────────────────────
    attacks_supplier = pd.DataFrame()
    if supplier_ids and conqueror_needs is not None:
        _s = (
            attacks_with_angle[attacks_with_angle["id_src"].isin(supplier_ids)]
            .merge(top5_ids[["id_src", "id", "is_top5"]], on=["id_src", "id"], how="left")
            .assign(is_top5=lambda d: d["is_top5"].fillna(False))
            .loc[lambda d: d["is_top5"]]
            .assign(target_is_supplier=lambda d: d["id"].isin(supplier_ids))
            .query("not target_is_supplier")
            .merge(
                conqueror_needs,
                left_on="id",
                right_on="id_src",
                how="right"
            )
            .query("(lowest_need - ships_min) * 1.5 < ships_sent")
            .query("ships_min * 0.75 < ships_sent < ships_min")
            .sort_values(["all_need", "ships_sent"], ascending=[False, True])
            .groupby(["id_src"], sort=False).first().reset_index()
        )
        attacks_supplier = _s

    # ── Combine and emit ─────────────────────────────────────────────────────
    parts = [df for df in [attacks_conqueror, attacks_conqueror_2, attacks_supplier] if df is not None and not df.empty]
    if not parts:
        return moves

    attacks = pd.concat(parts, ignore_index=True)
    print("Currently using testing _04_score_and_decide")
    for _, row in attacks.iterrows():
        print(f"From {row['id_src']}, To {row['id']} at step {row['step']} "
            f"with {row['ships_sent']} ships (target has min {row['ships_min']})")

    moves += attacks[["id_src", "final_angle", "ships_sent"]].values.tolist()
    return moves



## Test 01 — 2 Conquerors, 1 Enemy

Two blue planets (30 ships each) attack one red enemy (50 ships).
Expected: both conquerors should attack the enemy.

In [4]:
obs01 = Obs(
    planets=[
        [0, 0,  5.0,  5.0, 1 + math.log(3), 30, 3],  # Conqueror 1 (dist≈63.6, not orbiting)
        [1, 0,  5.0, 15.0, 1 + math.log(3), 30, 3],  # Conqueror 2 (dist≈57.0, not orbiting)
        [2, 1, 15.0, 15.0, 1 + math.log(3), 31, 2],  # Enemy       (dist≈42.4, not orbiting)
    ],
    angular_velocity=0.05,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,2.098612,30,3,0,fix
1,0,1,5.0,15.0,2.098612,30,3,0,fix
2,0,2,15.0,15.0,2.098612,31,2,1,fix
3,1,0,5.0,5.0,2.098612,33,3,0,fix
4,1,1,5.0,15.0,2.098612,33,3,0,fix
5,1,2,15.0,15.0,2.098612,33,2,1,fix
6,2,0,5.0,5.0,2.098612,36,3,0,fix
7,2,1,5.0,15.0,2.098612,36,3,0,fix
8,2,2,15.0,15.0,2.098612,35,2,1,fix
9,3,0,5.0,5.0,2.098612,39,3,0,fix


In [5]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,1.570796,10.000000,10.000000,7.901388,8.524700,0.000000e+00,0.161830,1.408966,1.732626,1.570796
1,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,1.570796,10.000000,10.000000,7.901388,8.556020,0.000000e+00,0.164822,1.405974,1.735618,1.570796
2,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,1.570796,10.000000,10.000000,7.901388,8.586829,0.000000e+00,0.167626,1.403170,1.738422,1.570796
3,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,1.570796,10.000000,10.000000,7.901388,8.617142,0.000000e+00,0.170258,1.400539,1.741054,1.570796
4,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,1.570796,10.000000,10.000000,7.901388,8.646976,0.000000e+00,0.172731,1.398065,1.743527,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.785398,14.142136,14.142136,15.244281,16.240748,1.217068e-01,0.000000,0.663691,0.907105,0.785398
668,1,0,5.0,15.0,2.098612,30,3,fix,0,11,...,-1.570796,10.000000,10.000000,11.198612,12.098612,1.629649e-01,0.000000,4.549424,4.875354,-1.570796
669,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.785398,14.142136,14.142136,12.628972,13.787901,1.088624e-01,0.148268,0.637130,0.933666,0.785398
670,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.785398,14.142136,14.142136,12.043523,12.198612,2.107342e-08,0.060291,0.725108,0.845689,0.785398


In [6]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.524700,0.000000e+00,0.161830,1.408966,1.732626,1.570796,1.570796
1,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.556020,0.000000e+00,0.164822,1.405974,1.735618,1.570796,1.570796
2,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.586829,0.000000e+00,0.167626,1.403170,1.738422,1.570796,1.570796
3,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.617142,0.000000e+00,0.170258,1.400539,1.741054,1.570796,1.570796
4,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.646976,0.000000e+00,0.172731,1.398065,1.743527,1.570796,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,14.142136,14.142136,15.244281,16.240748,1.217068e-01,0.000000,0.663691,0.907105,0.785398,0.785398
668,1,0,5.0,15.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,11.198612,12.098612,1.629649e-01,0.000000,4.549424,4.875354,-1.570796,-1.570796
669,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,14.142136,14.142136,12.628972,13.787901,1.088624e-01,0.148268,0.637130,0.933666,0.785398,0.785398
670,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,14.142136,14.142136,12.043523,12.198612,2.107342e-08,0.060291,0.725108,0.845689,0.785398,0.785398


In [7]:
action01 = _04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 30)
make_animation(snaps01, title='Test 01 — 2 Conquerors, 1 Enemy', interval=200)

Currently using testing _04_score_and_decide
From 1, To 2 at step 3 with 24 ships (target has min 30)
Action: [[1.0, 0.0, 24.0]]


## Test 02 — 2 Conquerors, 2 Neutrals

Two blue planets (30 ships each) face two neutral planets (50 and 60 ships).
Expected: each conqueror attacks the cheaper neutral target.

In [8]:
obs02 = Obs(
    planets=[
        [0, 0,  5.0,  5.0, 1 + math.log(3), 30, 3],   # Conqueror 1 (dist≈63.6, not orbiting)
        [1, 0,  5.0, 20.0, 1 + math.log(3), 30, 3],   # Conqueror 2 (dist≈57.4, not orbiting)
        [2, -1, 15.0,  5.0, 1 + math.log(3), 50, 2],  # Neutral 1   (dist≈42.7, not orbiting)
        [3, -1, 15.0, 20.0, 1 + math.log(3), 60, 2],  # Neutral 2   (dist≈42.4, not orbiting)
    ],
    angular_velocity=0.05,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.000000,5.000000,2.098612,30,3,0,fix
1,0,1,5.000000,20.000000,2.098612,30,3,0,fix
2,0,2,15.000000,5.000000,2.098612,50,2,-1,fix
3,0,3,15.000000,20.000000,2.098612,60,2,-1,moving
4,1,0,5.000000,5.000000,2.098612,33,3,0,fix
5,1,1,5.000000,20.000000,2.098612,33,3,0,fix
6,1,2,15.000000,5.000000,2.098612,50,2,-1,fix
7,1,3,15.000000,20.000000,2.098612,60,2,-1,moving
8,2,0,5.000000,5.000000,2.098612,36,3,0,fix
9,2,1,5.000000,20.000000,2.098612,36,3,0,fix


In [9]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.000000,10.000000,10.000000,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,0.000000
1,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.000000,10.000000,10.000000,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,0.000000
2,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.000000,10.000000,10.000000,7.901388,8.524700,0.000000,0.161830,6.121355,0.161830,0.000000
3,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.000000,10.000000,10.000000,7.901388,8.617142,0.000000,0.170258,6.112928,0.170258,0.000000
4,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.000000,10.000000,10.000000,7.901388,8.705271,0.000000,0.177251,6.105935,0.177251,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,-0.432130,26.914663,29.219277,26.971278,29.723797,0.077882,0.069136,5.773605,5.929369,-0.431914
963,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,0.103752,24.729637,26.676768,23.378342,25.731645,0.066792,0.071532,0.032219,0.218549,0.127755
964,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,-0.432005,26.914663,28.511593,28.579116,30.610202,0.046091,0.000125,5.805396,5.897578,-0.431852
965,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,-0.431876,26.914663,27.818778,28.759594,29.917377,0.035952,0.000254,5.815535,5.887439,-0.431787


In [10]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,0.000000,0.000000
1,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,0.000000,0.000000
2,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.524700,0.000000,0.161830,6.121355,0.161830,0.000000,0.000000
3,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.617142,0.000000,0.170258,6.112928,0.170258,0.000000,0.000000
4,0,0,5.0,5.0,2.098612,30,3,fix,0,11,...,10.000000,10.000000,7.901388,8.705271,0.000000,0.177251,6.105935,0.177251,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,26.914663,29.219277,26.514036,29.215749,0.077134,0.071843,5.774353,5.928621,-0.431914,-0.431914
933,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,26.914663,29.219277,26.971278,29.723797,0.077882,0.069136,5.773605,5.929369,-0.431914,-0.431914
934,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,26.914663,28.511593,28.579116,30.610202,0.046091,0.000125,5.805396,5.897578,-0.431852,-0.431852
935,1,0,5.0,20.0,2.098612,30,3,fix,0,11,...,26.914663,27.818778,28.759594,29.917377,0.035952,0.000254,5.815535,5.887439,-0.431787,-0.431787


In [11]:
action02 = _04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 30)
make_animation(snaps02, title='Test 02 — 2 Conquerors, 2 Neutrals', interval=200)

Currently using testing _04_score_and_decide
From 0, To 2 at step 3 with 20 ships (target has min 30)
Action: [[0.0, 0.0, 20.0]]


## Test 04 — 1 Supplier, 2 Conquerors, 1 Enemy

Three blue planets: one rich supplier (100 ships), two conquerors (30 and 20 ships). One red enemy (50 ships).
Expected: supplier reinforces the weaker conqueror; conquerors attack the enemy.

In [12]:
obs04 = Obs(
    planets=[
        [0, 0,  5.0,  5.0, 1 + math.log(3), 100, 5],  # Supplier    (dist≈63.6, not orbiting)
        [1, 0, 15.0,  5.0, 1 + math.log(3),  30, 3],  # Conqueror 1 (dist≈57.3, not orbiting)
        [2, 1, 25.0,  5.0, 1 + math.log(3),  35, 2],  # Enemy       (dist≈42.7, not orbiting)
        [3, 0,  5.0, 10.0, 1 + math.log(3),  30, 3],  # Conqueror 2 (dist≈57.4, not orbiting)
    ],
    angular_velocity=0.05,
)
df_s04, pd04 = StrategyPipeline._01_get_obs_dataframe(obs04, step=0, num_agents=2)
df_s04

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,5.0,5.0,2.098612,100,5,0,fix
1,0,1,15.0,5.0,2.098612,30,3,0,fix
2,0,2,25.0,5.0,2.098612,35,2,1,fix
3,0,3,5.0,10.0,2.098612,30,3,0,fix
4,1,0,5.0,5.0,2.098612,105,5,0,fix
5,1,1,15.0,5.0,2.098612,33,3,0,fix
6,1,2,25.0,5.0,2.098612,37,2,1,fix
7,1,3,5.0,10.0,2.098612,33,3,0,fix
8,2,0,5.0,5.0,2.098612,110,5,0,fix
9,2,1,15.0,5.0,2.098612,36,3,0,fix


In [13]:
pa04 = StrategyPipeline._02_get_all_opportunities(df_s04, pd04, player_id=0)
pa04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,1.570796,5.000000,5.000000,2.901388,3.198612,0.000000,0.270041,1.300756,1.840837,1.570796
1,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,1.570796,5.000000,5.000000,2.901388,5.022318,0.000000,0.421887,1.148909,1.992683,1.570796
2,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,1.570796,5.000000,5.000000,2.901388,5.044851,0.000000,0.420856,1.149940,1.991652,1.570796
3,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,1.570796,5.000000,5.000000,2.901388,5.066837,0.000000,0.419809,1.150987,1.990606,1.570796
4,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,1.570796,5.000000,5.000000,2.901388,5.088304,0.000000,0.418749,1.152048,1.989545,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1945,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,-0.244979,20.615528,20.615528,19.271312,21.168278,0.080877,0.096950,5.941256,6.135157,-0.244979
1946,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,-0.244979,20.615528,20.615528,19.858866,21.821117,0.096780,0.081012,5.941426,6.134987,-0.244979
1947,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,-0.244979,20.615528,20.615528,20.402100,22.424709,0.101842,0.049468,5.936364,6.140049,-0.244979
1948,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,-0.244979,20.615528,20.615528,18.630980,20.456798,0.034824,0.101943,5.936263,6.140150,-0.244979


In [14]:
safe04 = StrategyPipeline._03_filter_collision(pa04)
safe04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,5.000000,5.000000,2.901388,3.198612,0.000000,0.270041,1.300756,1.840837,1.570796,1.570796
1,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,5.000000,5.000000,2.901388,5.022318,0.000000,0.421887,1.148909,1.992683,1.570796,1.570796
2,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,5.000000,5.000000,2.901388,5.044851,0.000000,0.420856,1.149940,1.991652,1.570796,1.570796
3,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,5.000000,5.000000,2.901388,5.066837,0.000000,0.419809,1.150987,1.990606,1.570796,1.570796
4,0,0,5.0,5.0,2.098612,100,5,fix,0,11,...,5.000000,5.000000,2.901388,5.088304,0.000000,0.418749,1.152048,1.989545,1.570796,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,20.615528,20.615528,19.271312,21.168278,0.080877,0.096950,5.941256,6.135157,-0.244979,-0.244979
1618,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,20.615528,20.615528,19.858866,21.821117,0.096780,0.081012,5.941426,6.134987,-0.244979,-0.244979
1619,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,20.615528,20.615528,20.402100,22.424709,0.101842,0.049468,5.936364,6.140049,-0.244979,-0.244979
1620,3,0,5.0,10.0,2.098612,30,3,fix,0,11,...,20.615528,20.615528,18.630980,20.456798,0.034824,0.101943,5.936263,6.140150,-0.244979,-0.244979


In [15]:
action04 = _04_score_and_decide(safe04, player_id=0)
print("Action:", action04)
snaps04 = simulate_with_action(copy.deepcopy(obs04), action04, 30)
make_animation(snaps04, title='Test 04 — 1 Supplier, 2 Conquerors, 1 Enemy', interval=200)

Currently using testing _04_score_and_decide
From 1, To 2 at step 3 with 20 ships (target has min 30)
From 0, To 3 at step 1 with 76 ships (target has min 100)
Action: [[1.0, 0.0, 20.0], [0.0, 1.5707963267948966, 76.0]]
